７個のアイテムの0-1ナップサック問題

In [ ]:
# 7個のアイテムの0-1ナップサック問題データ
weights = [2, 3, 5, 7, 1, 4, 1]  # 各アイテムの重さ
values = [10, 5, 15, 7, 6, 18, 3] # 各アイテムの価値
W = 15                            # ナップサックの容量
N = 7                             # アイテムの総数

max_value = 0
best_combination = []

# DPを使わず、ビット全探索ですべての組み合わせ(2^7 = 128通り)を調べる
for i in range(1 << N):
    current_weight = 0
    current_value = 0
    current_combination = []

    for j in range(N):
        # i番目の組み合わせにおいて、j番目のアイテムを選ぶかどうかをビット判定
        if (i >> j) & 1:
            current_weight += weights[j]
            current_value += values[j]
            current_combination.append(j + 1) # アイテム番号(1始まり)を記録

    # 容量オーバーしておらず、かつこれまでの最大価値を上回った場合、記録を更新
    if current_weight <= W and current_value > max_value:
        max_value = current_value
        best_combination = current_combination

# 結果の出力
print(f"--- 問題設定 ---")
print(f"ナップサックの容量: {W}")
for i in range(N):
    print(f"アイテム {i+1}: 重さ {weights[i]:2d}, 価値 {values[i]:2d}")

print(f"\n--- 厳密解（全探索による結果） ---")
print(f"最大価値: {max_value}")
print(f"選ばれたアイテム: {best_combination}")

# 検算用の出力
total_weight = sum(weights[x-1] for x in best_combination)
print(f"合計の重さ: {total_weight} (容量 {W} 以下であることを確認)")

7～15個のアイテムでの0-1ナップサック問題

In [ ]:
import time
import random

# 結果を毎回再現できるように乱数のシード（種）を固定
random.seed(42)

print("--- 0-1ナップサック問題 全探索（N=7〜15） ---")
print("※「選ばれたアイテム」は1番目から数えた番号です。\n")

# Nを7から15まで増やすループ
for N in range(7, 16):
    # アイテムの重さ(1〜20)と価値(10〜100)をランダムに生成
    weights = [random.randint(1, 20) for _ in range(N)]
    values = [random.randint(10, 100) for _ in range(N)]
    W = 50  # ナップサックの容量を50に固定

    max_value = 0
    best_combination = []

    # 処理時間の計測開始
    start_time = time.perf_counter()

    # ビット全探索 (2^N 通りの組み合わせをすべて試す)
    for i in range(1 << N):
        current_weight = 0
        current_value = 0
        current_combination = []

        for j in range(N):
            # j番目のアイテムが選ばれているか判定
            if (i >> j) & 1:
                current_weight += weights[j]
                current_value += values[j]
                current_combination.append(j + 1) # アイテム番号(1始まり)を記録

        # 容量オーバーしておらず、これまでの最大価値を上回った場合、記録を更新
        if current_weight <= W and current_value > max_value:
            max_value = current_value
            best_combination = current_combination

    # 処理時間の計測終了
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    # 結果の出力
    print(f"N = {N:2d} | 処理時間: {elapsed_time:.6f} 秒 | 最大価値: {max_value:3d} | 選ばれたアイテム: {best_combination}")

グラフも描いてみる。

In [ ]:
import time
import random
import matplotlib.pyplot as plt

# 比較するアイテム数のリスト (7から15)
N_list = list(range(7, 16))
W = 50 # ナップサックの容量

times_bf = []
times_dp = []
results = []

# 再現性確保のため乱数シードを固定
random.seed(42)

for N in N_list:
    weights = [random.randint(1, 20) for _ in range(N)]
    values = [random.randint(10, 100) for _ in range(N)]

    # -------------------------
    # 1. 全探索 (Brute-Force)
    # -------------------------
    start_bf = time.perf_counter()
    max_val_bf = 0
    best_comb_bf = []

    for i in range(1 << N):
        cw, cv = 0, 0
        comb = []
        for j in range(N):
            if (i >> j) & 1:
                cw += weights[j]
                cv += values[j]
                comb.append(j + 1)
        if cw <= W and cv > max_val_bf:
            max_val_bf = cv
            best_comb_bf = comb

    time_bf = time.perf_counter() - start_bf
    times_bf.append(time_bf)

    # -------------------------
    # 2. 動的計画法 (DP)
    # -------------------------
    start_dp = time.perf_counter()

    # DPテーブル (アイテム数+1 x 容量+1)
    dp = [[0] * (W + 1) for _ in range(N + 1)]
    for i in range(1, N + 1):
        for w in range(W + 1):
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i-1][w], dp[i-1][w - weights[i-1]] + values[i-1])
            else:
                dp[i][w] = dp[i-1][w]

    max_val_dp = dp[N][W]

    # 選ばれたアイテムの復元 (バックトレース)
    best_comb_dp = []
    curr_w = W
    for i in range(N, 0, -1):
        # 現在の価値が1つ前のアイテムまでの価値と異なる場合、このアイテムは選ばれた
        if dp[i][curr_w] != dp[i-1][curr_w]:
            best_comb_dp.append(i) # 1-indexed
            curr_w -= weights[i-1]
    best_comb_dp.reverse() # 番号を昇順に直す

    time_dp = time.perf_counter() - start_dp
    times_dp.append(time_dp)

    results.append({
        'N': N, 'val_bf': max_val_bf, 'comb_bf': best_comb_bf, 'time_bf': time_bf,
        'val_dp': max_val_dp, 'comb_dp': best_comb_dp, 'time_dp': time_dp
    })

# --- グラフの描画 ---
plt.figure(figsize=(10, 6))
plt.plot(N_list, times_bf, label='Brute-Force', marker='o', color='red')
plt.plot(N_list, times_dp, label='DP', marker='s', color='green')

plt.title('Processing Time Comparison: Brute-Force vs Dynamic Programming')
plt.xlabel('Number of Items (N)')
plt.ylabel('Processing Time (Seconds)')
plt.legend()
plt.grid(True, linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()